# 9주차. 쉬운 베이지안 의사결정

## 오늘의 목표

이번 시간에는 베이즈 정리를 수학으로 증명하지 않는다. 대신 기획자가 **새 정보를 받았을 때 믿음을 어떻게 갱신하는지**를 그림과 사례로 익힌다.

수업이 끝나면 다음을 할 수 있어야 한다.

1. 점 추정과 확률 분포의 차이를 한 문장으로 설명한다.
2. 사전확률, 가능도, 사후확률을 일상 예시로 구분한다.
3. 의료 진단 사례에서 양성예측도가 왜 직관과 다른지 설명한다.
4. 베이지안 업데이트로 사후확률을 계산한다.
5. 사후확률을 결정 권고로 연결한다.

## 1. 점 추정 vs 확률 분포

기획자에게 매출을 물으면 흔히 **하나의 숫자**로 답한다.

> 내년 매출은 1억입니다.

이 답은 깔끔하지만 위험하다. **불확실성이 사라진다**. 만약 실제 매출이 7천만이 나오면 "틀린 예측"이 된다.

정직한 답은 **분포**로 답하는 것이다.

> 내년 매출은 8천만~1.2억 사이일 가능성이 높습니다(80% 신뢰). 평균은 1억 정도입니다.

베이지안은 이 분포를 그대로 다룬다. 하나의 값이 아니라 **가능한 값들의 확률**을 다룬다.

| 사고방식 | 표현 | 장점 | 단점 |
|---------|------|------|------|
| 점 추정 | "매출 1억" | 단순, 결정 빠름 | 불확실성 무시 |
| 확률 분포 | "8천만~1.2억(80%)" | 위험 인식 | 약간의 학습 필요 |

## 2. 사전·가능도·사후, 한 문장으로

베이즈 정리는 한 줄이다. 수식은 외우지 않는다. 의미만 기억하면 된다.

```
사후확률  =  가능도  ×  사전확률
             ───────────────────────
                  정규화 상수
```

**일상의 예시**

친구가 늦었다고 가정하자.

- **사전확률 (Prior)**: 평소 그 친구는 80% 시간 약속을 지킨다 → 늦을 확률 20%
- **가능도 (Likelihood)**: 오늘은 비가 와서 "비 오면 늦는 편"이라는 사실이 강하게 작동
- **사후확률 (Posterior)**: 비가 왔다는 정보를 본 다음, 늦을 확률은 50%로 올라감

한 줄 요약:

> 사전확률은 '내가 처음에 가진 믿음', 가능도는 '데이터가 그 가설과 얼마나 잘 맞는가', 사후확률은 '둘을 합친 갱신된 믿음'이다.

## 3. 의료 진단 예시: 직관이 틀리는 순간

베이즈 정리의 위력을 보여주는 고전 예시다. 학생이 기억해야 할 핵심 사례다.

**상황**

- 어떤 병의 유병률(전체 인구의 환자 비율) = **1%**
- 검사의 민감도(환자를 환자라고 맞힐 확률) = **99%**
- 검사의 특이도(건강한 사람을 건강하다고 맞힐 확률) = **95%**

**질문**: 어떤 사람이 이 검사에서 양성으로 나왔다. 이 사람이 실제로 환자일 확률은?

직관적으로는 "99%니까 거의 환자"라고 생각하기 쉽다. 실제로는 훨씬 낮다.

In [ ]:
# 기본 라이브러리
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import beta

In [ ]:
# 1만 명을 가정한 빈도표 (베이즈 정리 직관 버전)
population = 10_000
prevalence = 0.01
sensitivity = 0.99
specificity = 0.95

sick = int(population * prevalence)
healthy = population - sick

true_positive = int(sick * sensitivity)
false_negative = sick - true_positive
false_positive = int(healthy * (1 - specificity))
true_negative = healthy - false_positive

table = pd.DataFrame(
    [[true_positive, false_negative, sick],
     [false_positive, true_negative, healthy],
     [true_positive + false_positive, false_negative + true_negative, population]],
    columns=["Test Positive", "Test Negative", "Total"],
    index=["Sick", "Healthy", "Total"],
)
table

In [ ]:
# 양성예측도 = P(Sick | Test Positive)
ppv = true_positive / (true_positive + false_positive)
print(f"검사가 양성일 때 실제 환자일 확률 = {ppv:.1%}")
print(f"즉, 양성 결과의 약 {1 - ppv:.0%}는 거짓 양성이다.")

**해석**

검사 정확도가 99%여도 양성예측도는 약 17%다. 왜?

- 환자 100명 중 99명을 맞히지만, 그 수 자체가 작다.
- 건강한 9,900명 중 5%(495명)가 거짓 양성이 된다.
- 양성 진단을 받은 사람 중 실제 환자 비율은 **99 / (99 + 495) ≈ 17%**.

기획자에게 주는 교훈:

> **드문 사건일수록 사전확률(유병률)을 무시하면 의사결정이 틀린다.** 정책 효과 검증, 사기 탐지, AI 위험 평가에서 모두 같다.

In [ ]:
# 유병률에 따라 양성예측도가 어떻게 바뀌는지 본다
prevalence_grid = np.linspace(0.001, 0.5, 50)
ppv_grid = (prevalence_grid * sensitivity) / (
    prevalence_grid * sensitivity + (1 - prevalence_grid) * (1 - specificity)
)

plt.figure(figsize=(7, 4))
plt.plot(prevalence_grid, ppv_grid, color="black")
plt.axvline(0.01, color="gray", linestyle="--", linewidth=0.8)
plt.xlabel("Prevalence (사전확률)", color="black")
plt.ylabel("Positive Predictive Value", color="black")
plt.title("Prior shapes the answer", color="black")
plt.tight_layout()
plt.show()

## 4. 베이지안 업데이트: 정책 시범사업 사례

사례: **신규 청년 일자리 정책 시범 사업**

- 사전 믿음: "성공 확률은 30% 정도" — 근거가 약하므로 사전을 약하게 잡음.
- 새 정보: 시범 사업 10건 중 7건이 성공.
- 질문: 이 정보 이후 성공 확률 추정은 어떻게 바뀌는가?

확률 분포로 표현하기 위해 **베타 분포**를 쓴다. 베타 분포는 0~1 사이의 확률을 표현하기 좋다.

**매뉴얼 (외울 필요 없음, 따라 하면 됨)**

1. 사전을 `Beta(α, β)`로 적는다. α = 사전 성공, β = 사전 실패.
2. 새 데이터: 성공 s건, 실패 f건.
3. 사후 = `Beta(α + s, β + f)` ← 끝.

사전: `Beta(3, 7)` → 평균 0.30 (성공 3, 실패 7로 약하게 믿음)

In [ ]:
prior_alpha, prior_beta = 3, 7
successes, failures = 7, 3

post_alpha = prior_alpha + successes
post_beta = prior_beta + failures

x = np.linspace(0, 1, 200)

plt.figure(figsize=(7, 4))
plt.plot(x, beta.pdf(x, prior_alpha, prior_beta), color="gray", linestyle="--", label="Prior Beta(3, 7)")
plt.plot(x, beta.pdf(x, post_alpha, post_beta), color="black", label=f"Posterior Beta({post_alpha}, {post_beta})")
plt.xlabel("Probability of success", color="black")
plt.ylabel("Density", color="black")
plt.title("Prior vs Posterior", color="black")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Prior mean    = {prior_alpha / (prior_alpha + prior_beta):.2f}")
print(f"Posterior mean= {post_alpha / (post_alpha + post_beta):.2f}")

## 5. 사전이 강하면 어떻게 되는가

사전을 "성공 3건, 실패 7건"으로 잡았을 때(약한 믿음)와 "성공 30건, 실패 70건"으로 잡았을 때(강한 믿음)는 같은 30% 평균이라도 결과가 다르다.

기획자의 질문:

> 내가 이 결정에 얼마나 강하게 믿고 있는가? 데이터가 그 믿음을 흔들 만큼 충분한가?

In [ ]:
# 같은 데이터(7/10)를 받아도 사전 강도에 따라 사후가 다르다
scenarios = [
    ("Weak prior Beta(3, 7)", 3, 7),
    ("Strong prior Beta(30, 70)", 30, 70),
]

plt.figure(figsize=(7, 4))
for label, a, b in scenarios:
    pa, pb = a + 7, b + 3
    plt.plot(x, beta.pdf(x, pa, pb), label=f"{label} → mean={pa / (pa + pb):.2f}")
plt.axvline(0.7, color="gray", linestyle=":", alpha=0.7)
plt.xlabel("Probability of success", color="black")
plt.ylabel("Density", color="black")
plt.title("Same data, different priors", color="black")
plt.legend()
plt.tight_layout()
plt.show()

## 6. 데이터가 쌓이면 결국 데이터가 이긴다

데이터가 적을 때는 사전이 결과에 큰 영향을 준다. 데이터가 쌓이면 사전이 어떤 모양이든 **사후는 데이터에 가까워진다**.

이번 표는 진짜 성공률이 0.7이라고 가정하고, 표본 크기를 10에서 1000까지 늘리면서 사후 평균과 90% 신용 구간이 어떻게 변하는지 보여준다.

In [ ]:
rows = []
for n in [10, 50, 200, 1000]:
    s = int(round(n * 0.7))
    f = n - s
    a = prior_alpha + s
    b = prior_beta + f
    rows.append({
        "sample_size": n,
        "successes": s,
        "posterior_mean": round(a / (a + b), 3),
        "lower_5%": round(beta.ppf(0.05, a, b), 3),
        "upper_95%": round(beta.ppf(0.95, a, b), 3),
    })

pd.DataFrame(rows)

## 7. 바이브 코딩 프롬프트

오늘 실습에서 AI에게 요청할 문장이다.

```text
어떤 정책의 성공 확률에 대해 처음에는 30%라고 믿고 있었어.
시범 사업 10건 중 7건이 성공했어.
베타 분포를 사용해서 사전·사후 분포를 그리는 Python 코드를 만들어줘.
두 곡선을 같은 그래프에 흑백으로 그려주고, 사후 평균과 90% 신용 구간도 알려줘.
```

오류가 나면 오류 메시지를 그대로 복사해서 다시 묻는다.

```text
아래 Python 코드에서 오류가 났어. 오류 메시지를 보고 고쳐줘.
[오류 메시지]
```

## 8. 결정으로 연결하기

분석은 여기서 끝낼 수 있다. 기획은 한 단계 더 가야 한다.

**결정 매뉴얼**

1. 사후 평균을 계산한다.
2. 의사결정 임계값(threshold)을 정한다. 정책의 위험 허용 수준에 따라 다르다.
3. 사후 분포에서 임계값을 넘을 확률을 계산한다.
4. 권고를 작성한다: `Go`, `Hold`, `More data`.

위험이 큰 결정일수록 임계값을 높게 잡는다.

| 결정 유형 | 권장 임계값 |
|---------|-----------|
| 작은 시범 확장 | 0.5 |
| 본 사업 결정 | 0.7 |
| 사람 생명·안전 관련 | 0.9 |

In [ ]:
def make_decision(a, b, threshold=0.5):
    mean = a / (a + b)
    prob_above = 1 - beta.cdf(threshold, a, b)
    if prob_above >= 0.8:
        decision = "Go"
    elif prob_above >= 0.5:
        decision = "Hold (decide after more data)"
    else:
        decision = "Stop or redesign"
    return {
        "posterior_mean": round(mean, 3),
        "P(success > threshold)": round(prob_above, 3),
        "threshold": threshold,
        "decision": decision,
    }

decision_table = pd.DataFrame([
    make_decision(post_alpha, post_beta, t) for t in [0.3, 0.5, 0.7, 0.9]
])
decision_table[["threshold", "posterior_mean", "P(success > threshold)", "decision"]]

## 9. 학생 실습

관심 있는 정책 또는 사업의 의사결정을 하나 고른다.

예시:

- 신규 시범 사업 확대 여부
- 마케팅 캠페인 효과 검증
- 신제품 출시 결정
- 정책 도입 여부

**단계**

1. 결정해야 할 일을 한 문장으로 적는다.
2. 사전 믿음과 그 근거를 적는다 (예: "성공 40%, 비슷한 사례 5건 봄").
3. 관찰한 데이터를 적는다 (예: "시도 12건 중 9건 성공").
4. 사후를 계산한다.
5. 임계값을 정한다.
6. 결정을 권고한다.

AI에게 다음 프롬프트를 입력하고, 나온 코드를 이 노트북에 붙여 넣어 실행한다.

```text
내 결정 주제는 [주제명]이야.
현재 성공 확률에 대한 내 사전 믿음은 [예: 40%]이고, 강도는 약하게(샘플 10개 정도)야.
지금까지 관찰한 데이터는 시도 [N]건 중 [S]건 성공이야.
베타 분포로 사후 분포를 계산하고, 사전과 사후를 비교하는 그래프를 그려줘.
그리고 임계값 [0.5]를 넘을 확률을 알려줘.
```

In [ ]:
# 여기에 AI가 만들어준 분석 코드를 붙여 넣으세요.

my_prior_alpha = 0   # 사전 성공 수
my_prior_beta = 0    # 사전 실패 수
my_successes = 0     # 관찰 성공 수
my_failures = 0      # 관찰 실패 수

# my_post_alpha = my_prior_alpha + my_successes
# my_post_beta = my_prior_beta + my_failures
# print(make_decision(my_post_alpha, my_post_beta, threshold=0.5))

## 10. 제출물

아래 표를 작성하여 제출한다.

| 항목 | 작성 내용 |
|------|----------|
| 결정해야 할 일 | |
| 사전 믿음 (확률·근거·강도) | |
| 관찰한 데이터 | |
| 사후 평균과 90% 신용 구간 | |
| 임계값과 그 이유 | |
| 결정 권고 (Go / Hold / Stop) | |
| 결정을 바꾸려면 추가로 모을 정보 | |

## 오늘의 핵심

베이지안은 정답을 맞히는 도구가 아니다. **새 정보로 믿음을 갱신하고, 결정을 바꿀 만큼 충분한 증거가 있는지 묻는** 사고방식이다. 의료 진단 사례에서 봤듯이 사전확률을 무시하면 직관이 크게 어긋난다.